# Polyline

An open curve of zero thickness through a list of vertices: a thin plate, a wire or a grid
electrode.

`Polyline(vertices)` encloses no area, so `contains` is always false and it takes no part in
Boolean shapes. Its geometry is its outline, one open curve from the first vertex to the last,
which `boundary_panels` and `crossings` use directly.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from simgeometry import Polyline
from simgeometry.crossings import crossings

deflector = Polyline([(0.0, 0.3), (0.5, 0.7), (1.0, 0.3)])
(curve,) = deflector.outlines()
panels = deflector.boundary_panels(max_spacing=0.1)
print("closed:", deflector.closed, " vertices:", curve.shape[1])
print("panels:", len(panels), " total length:", panels.length.sum())
assert not deflector.contains((0.5, 0.5))
assert np.isclose(panels.length.sum(), 2 * np.hypot(0.5, 0.4))

## Particle steps

A particle step that passes through the curve stops on it: `crossings` returns, for each step
from `start` to `end`, whether it hits, where, and the normal there. Here vertical steps rise
from below the deflector.

In [ ]:
x = np.linspace(-0.2, 1.2, 15)
start = np.stack((x, np.zeros_like(x)))
end = np.stack((x, np.ones_like(x)))
hits = crossings(deflector, start, end)
print("steps stopped:", int(hits.hit.sum()), "of", len(x), " analytic:", hits.exact)

fig, ax = plt.subplots(figsize=(5.5, 4), layout="constrained")
for i in range(len(x)):
    stop = hits.point[:, i] if hits.hit[i] else end[:, i]
    ax.plot(*np.stack((start[:, i], stop), axis=1), color="0.7", lw=1)
ax.plot(*curve, color="k", lw=2, label="Polyline")
ax.plot(*hits.point[:, hits.hit], "o", color="tab:red", label="crossings")
ax.quiver(
    *hits.point[:, hits.hit],
    *hits.normal[:, hits.hit],
    color="tab:red",
    scale=14,
    width=0.005,
)
ax.set(xlabel="x", ylabel="y", title="Steps stopped by a Polyline")
ax.set_aspect("equal")
ax.legend(loc="lower center")
plt.show()